In [1]:
import pdal
import matplotlib.pyplot as plt
import numpy as np
import json

In [2]:
input_file = '/Users/jon.christo/Research/point_cloud/Aerotas_Point_Clouds_and_Ortho_DJI_L2_400ft/Aerotas - Lidar Point Cloud - L2 400ft.laz'

In [4]:
pipeline_json = {
    "pipeline": [
        {
            "type": "readers.las",
            "filename": "{input_file}"
        },
        {
            "type": "filters.outlier",
            "method": "statistical",
            "mean_k": 8,           # Number of neighbors to consider
            "multiplier": 2.0      # Threshold for outlier detection
        },
        {
            "type": "filters.smrf",
            "slope": 0.2,
            "scalar": 1.2,
            "threshold": 0.45,
            "window": 16.0
        },
        {
            "type": "filters.range",
            "limits": "Classification[2:2]"  # Only keep ground
        },
        {
            "type": "writers.gdal",
            "filename": "output_dtm.tif",
            "resolution": 1.0,
            "output_type": "min",
            "gdaldriver": "GTiff"
        }
    ]
}

pipeline = pdal.Pipeline(json.dumps(pipeline_json))
pipeline.execute()

RuntimeError: Unable to open stream for '{input_file}' with error 'No such file or directory'

In [ ]:
# Create PDAL pipeline to read in the LiDAR data with no filtering
pipeline = pdal.Pipeline(f"""
[
    {{
        "type":"readers.las",
        "filename":"{input_file}"
    }},
    {{
        "type": "filters.smrf",
        "scalar": 1.2,
        "slope": 0.3,
        "threshold": 3.0,
        "window": 18
    }}
]
""")

In [ ]:
# Execute pipeline
pipeline.execute()

In [ ]:
# Retrieve the data
arrays = pipeline.arrays[0]

In [ ]:
ground = arrays[arrays['Classification'] == 2]

In [ ]:
print(ground.shape)

In [ ]:
fig = plt.figure(figsize=(14, 8))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(ground['X'], ground['Y'], ground['Z'],
                c=ground['Z'], cmap='terrain', s=0.1)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
plt.colorbar(sc, label='Elevation')
plt.title("SMRF Ground Points Only")
plt.tight_layout()
plt.show()

In [ ]:
# Extract the X, Y, Z coordinates (no filtering, so this includes all points)
x = arrays['X']
y = arrays['Y']
z = arrays['Z']

In [ ]:
x = x[0:1000000]
y = y[0:1000000]
z = z[0:1000000]

In [ ]:
# Create a 3D plot of the LiDAR points (all points including trees, ground, and other features)
fig, ax = plt.subplots(figsize=(20, 9))
ax = fig.add_subplot(111, projection='3d')

# Plot the LiDAR points (no filtering, all points shown)
ax.scatter(x, y, z, s=0.1, c=z, cmap='viridis')  # Color based on Z for elevation

# Label axes
ax.set_xlabel('X Coordinate')
ax.set_ylabel('Y Coordinate')
ax.set_zlabel('Z Coordinate')

# Show plot
plt.show()